In [1]:
import transformers

print("Transformers version:", transformers.__version__)

c:\Users\Umair Mohi UD Din\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 5.18.0


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Qwen tokenizer loaded successfully!")

Qwen tokenizer loaded successfully!


In [5]:
import torch

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:03<00:00, 90.39it/s] 


Model loaded successfully!


In [7]:
SYSTEM_PROMPT = """
You are a helpful and friendly AI assistant.

Your job is to:
- Understand the user's question.
- Give clear and simple answers.
- Use conversation memory when available.
- Use external knowledge when required.
- If you do not know something, say that you do not know.
- Do not make up information.
"""

print("System prompt created successfully!")

System prompt created successfully!


In [8]:
def ask_qwen(user_message):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer

print("Chatbot function created successfully!")

Chatbot function created successfully!


In [9]:
question = "What is Generative AI?"

answer = ask_qwen(question)

print("User:", question)
print("Bot:", answer)

User: What is Generative AI?
Bot: Generative AI refers to artificial intelligence (AI) technologies that can create new content or generate text based on input. This includes things like language translation, image generation, music composition, and more. Essentially, it allows machines to produce human-like outputs without being explicitly programmed for those tasks.


In [10]:
# while True:

#     question = input("You: ")

#     if question.lower() == "exit":
#         print("Chat ended.")
#         break

#     answer = ask_qwen(question)

#     print("Bot:", answer)

In [11]:
conversation_history = []

print("Internal memory created!")

Internal memory created!


In [12]:
def ask_qwen_with_memory(user_message):

    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ] + conversation_history

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

print("Memory-enabled chatbot created!")

Memory-enabled chatbot created!


In [13]:
print(ask_qwen_with_memory("My name is Meerab."))

Hello Meerab! I'm an AI language model designed to help with various tasks like answering questions, providing information, and even helping with writing stories. How can I assist you today?


In [14]:
print(ask_qwen_with_memory("What is my name?"))

Your name is Meerab.


In [15]:
import json
import os

MEMORY_FILE = "chat_memory.json"

print("External memory setup ready!")

External memory setup ready!


In [16]:
def save_memory():

    with open(MEMORY_FILE, "w", encoding="utf-8") as file:
        json.dump(
            conversation_history,
            file,
            ensure_ascii=False,
            indent=4
        )

    print("Memory saved successfully!")

In [17]:
def load_memory():

    global conversation_history

    if os.path.exists(MEMORY_FILE):

        with open(MEMORY_FILE, "r", encoding="utf-8") as file:
            conversation_history = json.load(file)

        print("Previous memory loaded!")

    else:
        conversation_history = []
        print("No previous memory found.")

In [18]:
save_memory()

Memory saved successfully!


In [19]:
load_memory()

print(conversation_history)

Previous memory loaded!
[{'role': 'user', 'content': 'My name is Meerab.'}, {'role': 'assistant', 'content': "Hello Meerab! I'm an AI language model designed to help with various tasks like answering questions, providing information, and even helping with writing stories. How can I assist you today?"}, {'role': 'user', 'content': 'What is my name?'}, {'role': 'assistant', 'content': 'Your name is Meerab.'}]


In [20]:
def route_question(user_message):

    knowledge_keywords = [
        "according to",
        "document",
        "pdf",
        "research",
        "dataset",
        "machine learning",
        "deep learning"
    ]

    message = user_message.lower()

    for keyword in knowledge_keywords:
        if keyword in message:
            return "knowledge"

    return "normal"


print("Workflow router created!")

Workflow router created!


In [21]:
question1 = "What is machine learning?"

question2 = "Hello, how are you?"

print("Question 1:", route_question(question1))
print("Question 2:", route_question(question2))

Question 1: knowledge
Question 2: normal


In [22]:
def chatbot_workflow(user_message):

    route = route_question(user_message)

    if route == "knowledge":
        return "This question will be handled using the external knowledge/RAG system."

    else:
        return ask_qwen_with_memory(user_message)


print("Chatbot workflow created!")

Chatbot workflow created!


In [23]:
question = "Hello, my name is Meerab."

response = chatbot_workflow(question)

print("Bot:", response)

Bot: Hello Meerab! Is there anything specific you'd like to talk about or ask me today?


In [24]:
question = "What is machine learning?"

response = chatbot_workflow(question)

print("Bot:", response)

Bot: This question will be handled using the external knowledge/RAG system.


In [25]:
def calculator(expression):
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except:
        return "I could not calculate this expression."

print(calculator("25 * 18"))

450


In [26]:
tools = {
    "calculator": calculator
}

print("Available tools:", list(tools.keys()))

Available tools: ['calculator']


In [27]:
def choose_tool(user_message):

    math_symbols = ["+", "-", "*", "/", "%"]

    if any(symbol in user_message for symbol in math_symbols):
        return "calculator"

    return "none"


print(choose_tool("25 * 18"))
print(choose_tool("What is artificial intelligence?"))

calculator
none


In [28]:
def calculator(expression):
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except:
        return "Calculation error."


def memory_tool():
    return conversation_history


def current_time_tool():
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


tools = {
    "calculator": calculator,
    "memory": memory_tool,
    "current_time": current_time_tool
}

print("Available tools:")
for tool_name in tools:
    print("-", tool_name)

Available tools:
- calculator
- memory
- current_time


In [29]:
import re

def extract_math_expression(message):
    match = re.search(r"\d+(?:\s*[\+\-\*/%]\s*\d+)+", message)

    if match:
        return match.group(0)

    return None


def chatbot_workflow(user_message):

    tool_name = choose_tool(user_message)

    if tool_name == "calculator":

        expression = extract_math_expression(user_message)

        if expression:
            result = calculator(expression)
            return f"The answer is {result}."

        return "Please provide a mathematical expression."

    elif tool_name == "current_time":

        result = current_time_tool()

        return f"The current date and time is {result}."

    else:

        return ask_qwen_with_memory(user_message)


print("Tool-based chatbot workflow created!")


Tool-based chatbot workflow created!


In [30]:
print("Test 1:")
print(chatbot_workflow("What is 25 * 18?"))

print("\nTest 2:")
print(chatbot_workflow("What is the current time?"))

print("\nTest 3:")
print(chatbot_workflow("What is Generative AI?"))

Test 1:
The answer is 450.

Test 2:
As of my last update, the current time is 12:30 PM.

Test 3:
Generative AI refers to artificial intelligence (AI) systems that generate content based on input data. These systems learn from examples and use them to create new, original content without being explicitly programmed to do so. Examples include chatbots, virtual assistants, and machine learning models used in customer service and advertising.


In [31]:
def chatbot_workflow_with_memory(user_message):

    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    tool_name = choose_tool(user_message)

    if tool_name == "calculator":

        expression = extract_math_expression(user_message)

        if expression:
            result = calculator(expression)
            answer = f"The answer is {result}."
        else:
            answer = "Please provide a mathematical expression."

    elif tool_name == "current_time":

        result = current_time_tool()
        answer = f"The current date and time is {result}."

    else:

        answer = ask_qwen_with_memory(user_message)

        return answer

    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer


print("Memory + Tools workflow created!")

Memory + Tools workflow created!


In [32]:
print("Test 1: Calculator")
print(chatbot_workflow_with_memory("What is 45 * 12?"))

print("\nTest 2: Current Time")
print(chatbot_workflow_with_memory("What is the current time?"))

print("\nTest 3: Qwen")
print(chatbot_workflow_with_memory("What is artificial intelligence?"))

print("\nTest 4: Memory")
print(chatbot_workflow_with_memory("My name is Meerab."))
print(chatbot_workflow_with_memory("What is my name?"))

Test 1: Calculator
The answer is 540.

Test 2: Current Time
The current time is 12:30 PM.

Test 3: Qwen
Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. It involves developing intelligent machines that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, problem-solving, and natural language processing. AI aims to enable computers to think, learn, and act like humans.

Test 4: Memory
That's great to hear! What would you like to know more about?
Your name is Meerab.


In [33]:
print("🤖 Chatbot started!")
print("Type 'exit' to stop the chat.\n")

while True:

    user_message = input("You: ")

    if user_message.lower() == "exit":
        print("Bot: Goodbye!")
        break

    response = chatbot_workflow_with_memory(user_message)

    print("Bot:", response)

🤖 Chatbot started!
Type 'exit' to stop the chat.

Bot: Goodbye!


In [34]:
!pip install streamlit

In [35]:
import streamlit as st

print("Streamlit installed successfully!")

Streamlit installed successfully!


In [36]:
import streamlit as st

st.set_page_config(
    page_title="My AI Chatbot",
    page_icon="🤖"
)

st.title("🤖 My AI Chatbot")
st.write("Ask me anything!")

user_message = st.chat_input("Type your message...")

if user_message:
    st.write("You:", user_message)

2026-10-12 15:26:16.168 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:26:16.176 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:26:16.476 
  command:

    streamlit run C:\Users\Umair Mohi UD Din\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-10-12 15:26:16.477 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:26:16.477 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:26:16.477 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:26:16.477 Thread 'MainThread': missing ScriptRunContext! This w

In [37]:
if "messages" not in st.session_state:
    st.session_state.messages = []

# Purani chat display karo
for message in st.session_state.messages:

    with st.chat_message(message["role"]):
        st.write(message["content"])


# New message
if user_message:

    # User message save
    st.session_state.messages.append({
        "role": "user",
        "content": user_message
    })

    with st.chat_message("user"):
        st.write(user_message)

2026-10-12 15:26:35.822 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:26:35.822 Session state does not function when running a script without `streamlit run`
2026-10-12 15:26:35.822 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:26:35.822 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:26:35.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [38]:
if user_message:

    bot_response = chatbot_workflow_with_memory(user_message)

    st.session_state.messages.append({
        "role": "assistant",
        "content": bot_response
    })

    with st.chat_message("assistant"):
        st.write(bot_response)

print("Chatbot backend connected with UI!")

Chatbot backend connected with UI!


In [39]:
import os

if os.path.exists("app.py"):
    print("✅ app.py created successfully!")
    print("Ready to launch the chatbot UI.")
else:
    print("❌ app.py not found.")

✅ app.py created successfully!
Ready to launch the chatbot UI.


In [40]:
import sqlite3
from datetime import datetime

In [41]:
DB_FILE = "chatbot.db"

In [42]:
def init_database():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            role TEXT NOT NULL,
            message TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()

In [43]:
init_database()

In [44]:
def save_message(role, message):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute(
        """
        INSERT INTO conversations (role, message, timestamp)
        VALUES (?, ?, ?)
        """,
        (role, message, datetime.now().isoformat())
    )

    conn.commit()
    conn.close()

In [45]:
def load_messages():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT role, message
        FROM conversations
        ORDER BY id
    """)

    messages = cursor.fetchall()

    conn.close()

    return [
        {
            "role": role,
            "content": message
        }
        for role, message in messages
    ]

In [46]:
messages = load_messages()

print(messages)

[{'role': 'user', 'content': "what's the current time in Pakistan?"}, {'role': 'assistant', 'content': '🕐 **Current time in Pakistan:** 11:30:06 AM, Wednesday, 07 October 2026'}, {'role': 'user', 'content': 'what do you know about me ?'}, {'role': 'assistant', 'content': "🧠 **What I remember from our conversation:**\n\n- what's the current time in Pakistan?\n- what do you know about me ?\n"}, {'role': 'user', 'content': '30*15+100'}, {'role': 'assistant', 'content': '🧮 **Calculation Result:** `550`'}, {'role': 'user', 'content': 'my name is meerab . my age is 22'}, {'role': 'assistant', 'content': "Great to meet you, Meerab! It seems like we're just getting started here. Is there anything specific you'd like to talk about or ask about?"}, {'role': 'user', 'content': "what's my name ?"}, {'role': 'assistant', 'content': 'Your name is Meerab.'}, {'role': 'user', 'content': "i completed my bachelor's in information technology ."}, {'role': 'assistant', 'content': "That's great to hear tha

In [47]:
import time

def stream_response(response):
    words = response.split()

    for word in words:
        yield word + " "
        time.sleep(0.03)

In [48]:
for word in stream_response("Hello! How can I help you today?"):
    print(word, end="", flush=True)

Hello! How can I help you today? 

In [49]:
import streamlit as st

st.title("🤖 My AI Chatbot")

if "messages" not in st.session_state:
    st.session_state.messages = load_messages()

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])

user_input = st.chat_input("Type your message...")

2026-10-12 15:27:56.772 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:27:56.772 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:27:56.772 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:27:56.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:27:56.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:27:56.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:27:56.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:27:56.785 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [50]:
if user_input:

    st.session_state.messages.append({
        "role": "user",
        "content": user_input
    })

    save_message("user", user_input)

    with st.chat_message("user"):
        st.write(user_input)

In [51]:
response = "Hello! I am your AI assistant. How can I help you?"

In [52]:
with st.chat_message("assistant"):

    st.write_stream(
        stream_response(response)
    )

st.session_state.messages.append({
    "role": "assistant",
    "content": response
})

save_message("assistant", response)

2026-10-12 15:28:15.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:28:15.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:28:15.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:28:15.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:28:15.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:28:15.545 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:28:15.546 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-12 15:28:15.547 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [53]:
import threading
from transformers import TextIteratorStreamer

In [54]:
stop_event = threading.Event()
is_generating = False
partial_response = ""

In [55]:
def generate_stream(messages):
    global is_generating, partial_response

    stop_event.clear()
    is_generating = True
    partial_response = ""

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = {
        **model_inputs,
        "streamer": streamer,
        "max_new_tokens": 512,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9
    }

    thread = threading.Thread(
        target=model.generate,
        kwargs=generation_kwargs
    )

    thread.start()

    for new_text in streamer:

        # STOP
        if stop_event.is_set():
            break

        partial_response += new_text

        yield new_text

    is_generating = False

In [56]:
def stop_generation():
    global is_generating

    if is_generating:
        stop_event.set()
        is_generating = False
        print("\n⏹️ Generation stopped.")
    else:
        print("⚠️ No generation is currently running.")

In [57]:
def resume_generation():
    global partial_response
    global is_generating

    if not partial_response:
        print("⚠️ Nothing to resume.")
        return

    if is_generating:
        print("⚠️ Generation is already running.")
        return

    previous_response = partial_response

    resume_messages = messages.copy()

    resume_messages.append({
        "role": "assistant",
        "content": previous_response
    })

    resume_messages.append({
        "role": "user",
        "content": (
            "Continue your previous response from where "
            "you stopped. Do not repeat the previous text."
        )
    })

    print("\n▶️ Resuming...\n")

    continued_response = ""

    for token in generate_stream(
        resume_messages
    ):
        continued_response += token
        print(
            token,
            end="",
            flush=True
        )

    partial_response = (
        previous_response +
        continued_response
    )

    if messages and messages[-1]["role"] == "assistant":
        messages[-1]["content"] = partial_response

In [60]:

messages = [ 
    {"role": "user", "content": "Hello, my name is Meerab."}
]

formatted_messages = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [61]:
full_response = ""

for token in generate_stream(formatted_messages):

    full_response += token

    print(
        token,
        end="",
        flush=True
    )

print()

As an AI language model, I don't have personal feelings or emotions like humans do. However, I'm happy to provide information about various topics and answer any questions you might have! If you have something specific in mind, feel free to ask, and I'll do my best to help you out.


In [63]:
try:
    full_response = ""
    print("Generating response (press stop button...")
    
    for token in generate_stream(formatted_messages):
        full_response += token
        print(token, end="", flush=True)
        
except KeyboardInterrupt:
    print("\n\n[Generation stopped from user side / Stopped]")

Generating response (press stop button...
As an AI language model, I don't have personal feelings or emotions like humans do. However, I can provide you with factual information about the topic you're interested in.

If you have any specific questions or topics you'd like to know more about, feel free to ask!

In [64]:

messages.append({"role": "assistant", "content": full_response})

formatted_messages = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

try:
    print("Resuming generation...")
    for token in generate_stream(formatted_messages):
        full_response += token
        print(token, end="", flush=True)
except KeyboardInterrupt:
    print("\n\n[Generation stop]")

Resuming generation...
As an AI language model, I don't have personal experiences or emotions like humans do, but I can provide you with some insights on how to approach and handle difficult conversations.

When dealing with someone who is upset about something, it's important to listen actively without interrupting. Try to understand their perspective and try to find common ground between both of your sides. It might be helpful to ask questions that show you're interested in what they have to say. For example, "Can you tell me more about the situation? What was the problem?" This can help build rapport and make the conversation feel more comfortable.

If the issue at hand involves emotions or personal matters, it may be helpful to seek support from a trusted friend or family member. They can offer emotional comfort and practical advice based on their own experience. Additionally, if the person is not willing to talk or seems resistant to change, consider reaching out to a mental healt

In [65]:

MAX_CONTEXT_MESSAGES = 10

if 'messages' in globals() and len(messages) > MAX_CONTEXT_MESSAGES:
    messages = [messages[0]] + messages[-MAX_CONTEXT_MESSAGES:]
    print(f"Context managed: Total messages ab {len(messages)} hain.")
else:
    print("Context window is under limit .")

Context window is under limit .


In [66]:

mcp_data = "Active MCP Server Data: [User session details & retrieved documents loaded]"

mcp_system_message = {
    "role": "system",
    "content": f"You are a helpful AI assistant. Use the following MCP context when answering:\n{mcp_data}"
}

if 'messages' not in globals():
    messages = []

if len(messages) > 0 and messages[0]["role"] == "system":
    messages[0] = mcp_system_message
else:
    messages.insert(0, mcp_system_message)

print("MCP Context successfully injected into system prompt!")

MCP Context successfully injected into system prompt!


In [67]:

formatted_messages = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

try:
    full_response = ""
    print("Generating response...\n")
    
    for token in generate_stream(formatted_messages):
        full_response += token
        print(token, end="", flush=True)
        
except KeyboardInterrupt:
    print("\n\n[Generation stopped from user side  / Stopped]")


if 'full_response' in locals() and full_response:
    messages.append({"role": "assistant", "content": full_response})

Generating response...

As an AI language model, I don't have personal experiences or emotions like humans do, but I can provide you with information and insights on various topics related to the universe, technology, culture, and more.

If you have any specific questions about these areas, feel free to ask, and I'll do my best to answer them accurately and thoughtfully!